In [21]:

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

try:
    movies = pd.read_csv('../data/movies.csv')
    tags = pd.read_csv('../data/tags.csv')
    print(f'Datasetet laddat: {len(movies)} filmer hittades.')
except FileNotFoundError:
    print('Fel: Hittade inte movies.csv eller tags.csv. Se till att filen ligger i samma mapp.')

movies['genre_text'] = movies['genres'].str.replace('|', '', regex=False)

tags['tag'] = tags['tag'].astype('str').str.lower().str.strip()
tags_grouped = (
    tags.groupby('movieId')['tag']
    .apply(lambda x: ' '.join(x.unique()))
    .reset_index()
    .rename(columns={'tag': 'tag_text'})
)

movies = movies.merge(tags_grouped, on='movieId', how='left')
movies['tag_text'] = movies['tag_text'].fillna('')
movies['combined_text'] = movies['genre_text'] + ' ' + movies['tag_text']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['combined_text'])

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(tfidf_matrix)

def get_recommendations(movies_title, n=5):
    matches = movies[movies['title'].str.contains(movies_title, case=False, na=False, regex=False)]

    if matches.empty:
        print(f'Ingen film hittades för "{movies_title}".')
        return None
    
    index = matches.index[0]
    distances, indices = model.kneighbors(tfidf_matrix[index], n_neighbors=n+1)

    rec_idx = indices[0][1:]
    rec_sim = 1 - distances[0][1:]


    return movies.iloc[rec_idx][['title', 'genres']].assign(
            similarity=rec_sim,
            genres=lambda df: df['genres'].str.replace('|', ', ', regex=False)
    ).reset_index(drop=True)


result = get_recommendations('Toy Story (1995)', n=5)
display(result)


Datasetet laddat: 86537 filmer hittades.


,title,genres,similarity
0,Toy Story 2 (1999),"Adventure, Animation, Children, Comedy, Fantasy",0.565441
1,Toy Story 4 (2019),"Adventure, Animation, Children, Comedy",0.551361
2,Toy Story 3 (2010),"Adventure, Animation, Children, Comedy, Fantas...",0.529842
3,Toy Story That Time Forgot (2014),"Animation, Children",0.404601
4,Toy Story Toons: Small Fry (2011),"Adventure, Animation, Children, Comedy, Fantasy",0.374922
